# Análise de Risco de Mercado nas Maiores Blue Chips da B3
### PETR4 · ITUB4 · VALE3 · ABEV3 · BPAC11 | 2018–2025

**Carteira:** PETR4 · ITUB4 · VALE3 · ABEV3 · BPAC11 — as 5 maiores da B3 por valor de mercado  
**Benchmark:** IBOV | **Período:** 2018–2025 | **Pesos:** iguais (20% cada)

| Etapa | Conteúdo |
|---|---|
| 1 | ETL — coleta e tratamento |
| 2 | Métricas de risco |
| 3 | Dashboard — 5 gráficos interativos |
| 4 | Conclusões |


## 0. Instalação

In [10]:
import sys
!{sys.executable} -m pip install yfinance pandas numpy plotly scipy -q

import plotly.io as pio
pio.renderers.default = "browser"

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm, kurtosis, skew, chi2
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

ATIVOS    = ["PETR4.SA", "ITUB4.SA", "VALE3.SA", "ABEV3.SA", "BPAC11.SA"]
NOMES     = ["Petrobras", "Itaú", "Vale", "Ambev", "BTG"]
BENCHMARK = "^BVSP"
START     = "2018-01-01"
END       = "2025-12-31"
CONF      = 0.95
WINDOW    = 252

CORES = {
    "PETR4.SA": "#1f77b4",
    "ITUB4.SA": "#2ca02c",
    "VALE3.SA": "#d62728",
    "ABEV3.SA": "#ff7f0e",
    "BPAC11.SA": "#9467bd",
    "IBOV":     "#333333",
}

CRISES = {
    "Covid (mar/20)":           "2020-03-23",
    "Crise Eleitoral (out/22)": "2022-10-30",
}

print("ok")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
ok


## 1. ETL — Coleta e Tratamento dos Dados

Retorno log diário:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

Retorno da carteira com pesos iguais:

$$r_{carteira,t} = \frac{1}{5}\sum_{i=1}^{5} r_{i,t}$$

O retorno acumulado a partir de retornos log é calculado como:

$$R_{acum} = e^{\sum r_t}$$


In [11]:
print("Baixando dados...")
tickers = ATIVOS + [BENCHMARK]
raw = yf.download(tickers, start=START, end=END,
                  auto_adjust=True, progress=False)

prices = raw["Close"].copy()
prices.index = pd.to_datetime(prices.index)
if prices.index.tz is not None:
    prices.index = prices.index.tz_localize(None)

prices = prices.loc[START:END].ffill(limit=1).dropna()
prices = prices.rename(columns={"^BVSP": "IBOV"})

returns = np.log(prices / prices.shift(1)).dropna()
returns = returns.loc[START:END]

PESOS = np.array([0.20] * 5)
port_returns = (returns[ATIVOS] * PESOS).sum(axis=1)

# retorno acumulado correto para retornos log: exp(soma dos retornos)
cum_ibov = np.exp(returns["IBOV"].cumsum())

returns.to_csv("data/processed/returns.csv")
port_returns.to_csv("data/processed/port_returns.csv")

print("Período:      {} -> {}".format(
    returns.index[0].date(), returns.index[-1].date()))
print("Observações:  {} dias úteis".format(len(returns)))
print()
print("Retorno anualizado por ativo:")
for a, n in zip(ATIVOS, NOMES):
    print("  {:<12} {:.2%}".format(n, returns[a].mean()*252))

Baixando dados...
Período:      2018-01-03 -> 2025-12-30
Observações:  1987 dias úteis

Retorno anualizado por ativo:
  Petrobras    24.66%
  Itaú         11.67%
  Vale         15.22%
  Ambev        -1.29%
  BTG          33.77%


## 2. Métricas de Risco

**VaR Histórico (95%):**
$$\text{VaR}_{hist} = -Q_{0.05}(r_t)$$

Perda máxima esperada em 95% dos dias. Sem premissa de distribuição — usa os dados como são.

**VaR Paramétrico (95%):**
$$\text{VaR}_{param} = -(\mu - z_{0.95}\,\sigma)$$

Assume distribuição normal dos retornos. Tende a subestimar o risco em períodos de crise.

**CVaR / ETL (95%):**
$$\text{CVaR} = -E[r_t \mid r_t \leq Q_{0.05}]$$

Perda média nos 5% piores dias. Sempre maior que o VaR. Métrica preferida pelos reguladores (Basileia III).

In [12]:
mu    = port_returns.mean()
sigma = port_returns.std()
z     = norm.ppf(CONF)

var_hist   = -port_returns.quantile(1 - CONF)
var_param  = -(mu - z * sigma)
tail       = port_returns[port_returns <= -var_hist]
cvar_hist  = -tail.mean()
cvar_param = -(mu - sigma * norm.pdf(z) / (1 - CONF))
sk         = skew(port_returns)
ku         = kurtosis(port_returns)

# retorno acumulado correto para retornos log
cum_port      = np.exp(port_returns.cumsum())
max_dd        = ((cum_port - cum_port.cummax()) / cum_port.cummax()).min()
retorno_total = cum_port.iloc[-1] - 1
vol_anual     = sigma * np.sqrt(252)
sharpe        = (mu * 252) / vol_anual

# backtesting com shift(1) para evitar look-ahead bias:
# o VaR do dia t e calculado com dados ate t-1
var_rolling = -port_returns.rolling(WINDOW).quantile(1 - CONF).shift(1)
bt = pd.DataFrame({"retorno": port_returns, "var": var_rolling}).dropna()
bt["excecao"] = bt["retorno"] < -bt["var"]

n_total    = len(bt)
n_excecoes = int(bt["excecao"].sum())
taxa_obs   = n_excecoes / n_total
taxa_esp   = 1 - CONF

# teste de Kupiec: avalia estatisticamente se a taxa de excecoes
# e compativel com o nivel de confianca do modelo
p_obs = n_excecoes / n_total
LR_kupiec = -2 * np.log(
    ((1 - taxa_esp)**(n_total - n_excecoes) * taxa_esp**n_excecoes) /
    ((1 - p_obs)**(n_total - n_excecoes) * p_obs**n_excecoes)
)
p_valor_kupiec = chi2.sf(LR_kupiec, df=1)
avaliacao = "Aceito" if p_valor_kupiec > 0.05 else "Rejeitado"

print("=" * 52)
print("  Métricas de Risco da Carteira (1 dia, 95%)")
print("=" * 52)
print("  Retorno Total:        {:.2%}".format(retorno_total))
print("  Volatilidade Anual:   {:.2%}".format(vol_anual))
print("  Sharpe Ratio:         {:.2f}".format(sharpe))
print("  VaR Histórico:        {:.2%}".format(var_hist))
print("  VaR Paramétrico:      {:.2%}".format(var_param))
print("  CVaR Histórico:       {:.2%}".format(cvar_hist))
print("  CVaR Paramétrico:     {:.2%}".format(cvar_param))
print("  Máx. Drawdown:        {:.2%}".format(max_dd))
print("  Skewness:             {:.2f}".format(sk))
print("  Kurtosis:             {:.2f}".format(ku))
print()
print("  Backtesting (Teste de Kupiec)")
print("  Exceções observadas:  {}/{} ({:.2%})".format(n_excecoes, n_total, taxa_obs))
print("  Exceções esperadas:   {:.2%}".format(taxa_esp))
print("  p-valor Kupiec:       {:.4f}".format(p_valor_kupiec))
print("  Resultado:            {} (modelo {} calibrado)".format(
    avaliacao, "bem" if avaliacao == "Aceito" else "mal"))
print("=" * 52)

  Métricas de Risco da Carteira (1 dia, 95%)
  Retorno Total:        276.34%
  Volatilidade Anual:   25.82%
  Sharpe Ratio:         0.65
  VaR Histórico:        2.25%
  VaR Paramétrico:      2.61%
  CVaR Histórico:       3.72%
  CVaR Paramétrico:     3.29%
  Máx. Drawdown:        -51.35%
  Skewness:             -1.17
  Kurtosis:             24.15

  Backtesting (Teste de Kupiec)
  Exceções observadas:  84/1735 (4.84%)
  Exceções esperadas:   5.00%
  p-valor Kupiec:       0.7608
  Resultado:            Aceito (modelo bem calibrado)


## 3. Dashboard

### Gráfico 1 — Distribuição de Retornos e Métricas de Risco

O histograma mostra a distribuição real dos retornos diários da carteira.
As linhas verticais marcam os limites de perda estimados por cada método.


In [13]:
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
pdf = norm.pdf(x * 100, mu * 100, sigma * 100) / 100
ret_plot = port_returns[(port_returns >= mu - 4*sigma) &
                        (port_returns <= mu + 4*sigma)]

fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=ret_plot * 100, nbinsx=70,
    histnorm="probability density",
    name="Retornos observados",
    marker_color="#5b9bd5", opacity=0.75))
fig1.add_trace(go.Scatter(
    x=x * 100, y=pdf, mode="lines",
    name="Distribuição Normal",
    line=dict(color="#d62728", width=2)))

for val, label, cor in [
    (var_param, "VaR Paramétrico ({:.2%})".format(var_param), "#e8892b"),
    (var_hist,  "VaR Histórico ({:.2%})".format(var_hist),    "#d4a017"),
    (cvar_hist, "CVaR ({:.2%})".format(cvar_hist),            "#7b2d8b"),
]:
    fig1.add_vline(x=-val*100, line_dash="dash", line_color=cor,
                   annotation_text=label,
                   annotation_position="top left",
                   annotation_font_size=10)

fig1.add_annotation(
    x=0.02, y=0.98, xref="paper", yref="paper",
    text="Skewness: {:.2f}<br>Kurtosis: {:.2f}".format(sk, ku),
    showarrow=False, align="left",
    bgcolor="#fffde7", bordercolor="#dddddd", borderwidth=1,
    font=dict(size=11))
fig1.update_layout(
    title="Distribuição de Retornos e Métricas de Risco",
    xaxis_title="Retorno Diário (%)",
    yaxis_title="Densidade de Probabilidade",
    legend=dict(orientation="h", y=-0.2),
    height=450, template="plotly_white")
fig1.show()

### Gráfico 2 — Retorno Acumulado por Ativo vs IBOV

Mostra quanto R$1 investido em cada ativo em 2018 valeria em 2025.
O IBOV serve como benchmark de comparação.


In [14]:
fig2 = go.Figure()
for ativo, nome in zip(ATIVOS, NOMES):
    # retorno acumulado correto para retornos log
    cum = np.exp(returns[ativo].cumsum())
    fig2.add_trace(go.Scatter(
        x=cum.index, y=cum.values,
        name=ativo.replace(".SA", ""),
        line=dict(color=CORES[ativo], width=1.8)))
fig2.add_trace(go.Scatter(
    x=cum_ibov.index, y=cum_ibov.values,
    name="IBOV (benchmark)",
    line=dict(color="#333333", width=2.5, dash="dash")))
for label, data in CRISES.items():
    fig2.add_vline(x=data, line_dash="dot", line_color="#999999",
                   opacity=0.8, annotation_text=label,
                   annotation_position="top",
                   annotation_font_size=10)
fig2.add_hline(y=1, line_dash="dot", line_color="#cccccc")
fig2.update_layout(
    title="Retorno Acumulado por Ativo vs IBOV",
    yaxis_title="Retorno (base 1,0)",
    legend=dict(orientation="h", y=-0.15),
    height=450, template="plotly_white")
fig2.show()

### Gráfico 3 — Drawdown Histórico: Carteira vs IBOV

O drawdown mede a queda em relação ao pico histórico.
Permite visualizar o impacto dos choques de mercado ao longo do tempo.


In [15]:
dd_port = (cum_port - cum_port.cummax()) / cum_port.cummax() * 100
dd_ibov = (cum_ibov - cum_ibov.cummax()) / cum_ibov.cummax() * 100

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=dd_port.index, y=dd_port.values,
    name="Carteira", fill="tozeroy",
    fillcolor="rgba(31,119,180,0.15)",
    line=dict(color="#1f77b4", width=2)))
fig3.add_trace(go.Scatter(
    x=dd_ibov.index, y=dd_ibov.values,
    name="IBOV (benchmark)",
    line=dict(color="#333333", width=2, dash="dash")))
for label, data in CRISES.items():
    fig3.add_vline(x=data, line_dash="dot", line_color="#999999",
                   opacity=0.8, annotation_text=label,
                   annotation_position="top",
                   annotation_font_size=10)
fig3.add_hline(y=0, line_color="#cccccc", line_width=1)
fig3.update_layout(
    title="Drawdown Histórico — Carteira vs IBOV",
    yaxis_title="Drawdown (%)",
    legend=dict(orientation="h", y=-0.15),
    height=400, template="plotly_white")
fig3.show()

### Gráfico 4 — Comparação de Métodos VaR

Compara as quatro estimativas de risco lado a lado.
O CVaR é sempre maior que o VaR por capturar o tamanho das perdas extremas.


In [16]:
metodos   = ["VaR Paramétrico", "VaR Histórico", "CVaR Histórico", "CVaR Paramétrico"]
valores   = [var_param*100, var_hist*100, cvar_hist*100, cvar_param*100]
cores_bar = ["#e8892b", "#d4a017", "#d62728", "#7b2d8b"]

fig4 = go.Figure(go.Bar(
    x=metodos, y=valores,
    marker_color=cores_bar,
    text=["{:.2f}%".format(v) for v in valores],
    textposition="outside",
    width=0.5))
fig4.update_layout(
    title="Comparação de Métodos VaR (1 dia, 95%)",
    yaxis_title="Perda Esperada (%)",
    yaxis=dict(range=[0, max(valores) * 1.3]),
    showlegend=False,
    height=400, template="plotly_white")
fig4.show()

### Gráfico 5 — Backtesting do VaR

Valida o modelo comparando as exceções observadas com as esperadas.
Para VaR a 95%, esperamos 5% de exceções.

O **Teste de Kupiec** verifica estatisticamente se a taxa observada é compatível
com o nível de confiança do modelo (H₀: modelo bem calibrado, rejeitado se p-valor < 0,05).


In [17]:
excecoes_acum = bt["excecao"].cumsum()
esperado_acum = pd.Series(
    np.arange(1, len(bt)+1) * taxa_esp, index=bt.index)

fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=bt.index, y=excecoes_acum,
    name="Observado ({:.1%})".format(taxa_obs),
    line=dict(color="#d62728", width=1.8),
    fill="tonexty", fillcolor="rgba(214,39,40,0.08)"))
fig5.add_trace(go.Scatter(
    x=bt.index, y=esperado_acum,
    name="Esperado (5%)",
    line=dict(color="#1f77b4", width=1.5, dash="dash")))

cor_av = "#2ca02c" if p_valor_kupiec > 0.05 else "#d62728"
kupiec_txt = "Kupiec: {} (p={:.3f})".format(avaliacao, p_valor_kupiec)
fig5.add_annotation(
    x=0.98, y=0.05, xref="paper", yref="paper",
    text=kupiec_txt, showarrow=False,
    font=dict(size=11, color=cor_av),
    xanchor="right")
fig5.update_layout(
    title="Backtesting do VaR (Exceções Acumuladas)",
    yaxis_title="Exceções Acumuladas",
    legend=dict(orientation="h", y=-0.15),
    height=400, template="plotly_white")
fig5.show()

## 4. Conclusões

Resumo dos principais resultados da análise.


In [18]:
# retorno acumulado correto para retornos log
retornos_totais = {
    a.replace(".SA",""): np.exp(returns[a].cumsum()).iloc[-1] - 1
    for a in ATIVOS
}
ranking = sorted(retornos_totais.items(), key=lambda x: x[1], reverse=True)
melhor  = ranking[0]
pior    = ranking[-1]

var_por_ativo = {
    a.replace(".SA",""): -returns[a].quantile(1 - CONF)
    for a in ATIVOS
}
mais_arriscado = max(var_por_ativo, key=var_por_ativo.get)

drawdowns_crise = {}
for nome_c, data_c in CRISES.items():
    r = port_returns.loc[:data_c].iloc[-21:]
    drawdowns_crise[nome_c] = r.sum()
maior_choque = min(drawdowns_crise, key=drawdowns_crise.get)

print("=" * 58)
print("  Resumo da Carteira")
print("=" * 58)
print("  Retorno Acumulado      {:>10}".format("{:.1%}".format(retorno_total)))
print("  Volatilidade Anual     {:>10}".format("{:.1%}".format(vol_anual)))
print("  Sharpe Ratio           {:>10}".format("{:.2f}".format(sharpe)))
print("  VaR Histórico 95%      {:>10}".format("{:.2%}".format(var_hist)))
print("  CVaR Histórico 95%     {:>10}".format("{:.2%}".format(cvar_hist)))
print("  Máx. Drawdown          {:>10}".format("{:.1%}".format(max_dd)))
print("=" * 58)
print()
print("  Desempenho")
print("  - {} teve o maior retorno acumulado ({:.0%}).".format(
    melhor[0], melhor[1]))
print("  - {} teve o pior desempenho no período ({:.0%}).".format(
    pior[0], pior[1]))
print()
print("  Risco")
print("  - {} apresentou o maior VaR individual ({:.2%}).".format(
    mais_arriscado, var_por_ativo[mais_arriscado]))
print("  - VaR da carteira: {:.2%} | CVaR: {:.2%}".format(
    var_hist, cvar_hist))
print("  - Máximo drawdown da carteira: {:.1%}".format(max_dd))
print()
print("  Choques de mercado")
print("  - {} representou o maior choque do período.".format(
    maior_choque))
print()
print("  Validação do modelo (Teste de Kupiec)")
print("  - Exceções: {}/{} ({:.2%} vs {:.0%} esperado)".format(
    n_excecoes, n_total, taxa_obs, taxa_esp))
print("  - p-valor: {:.4f} — {} (modelo {} calibrado)".format(
    p_valor_kupiec, avaliacao,
    "bem" if avaliacao == "Aceito" else "mal"))

  Resumo da Carteira
  Retorno Acumulado          276.3%
  Volatilidade Anual          25.8%
  Sharpe Ratio                 0.65
  VaR Histórico 95%           2.25%
  CVaR Histórico 95%          3.72%
  Máx. Drawdown              -51.4%

  Desempenho
  - BPAC11 teve o maior retorno acumulado (1334%).
  - ABEV3 teve o pior desempenho no período (-10%).

  Risco
  - BPAC11 apresentou o maior VaR individual (3.80%).
  - VaR da carteira: 2.25% | CVaR: 3.72%
  - Máximo drawdown da carteira: -51.4%

  Choques de mercado
  - Covid (mar/20) representou o maior choque do período.

  Validação do modelo (Teste de Kupiec)
  - Exceções: 84/1735 (4.84% vs 5% esperado)
  - p-valor: 0.7608 — Aceito (modelo bem calibrado)
